# Weight-Space Engineering — Interactive Quickstart (`Lab 01`)
**By Aguentil H.** · [Read Chapter 1 Free](https://weight-space-engineering.weight-space-engineering.workers.dev/chapter-1/) · [Complete 94-Page Book + All 8 Labs](https://whop.com/weight-space-engineering/weight-space-engineering-handbook-python-lab-suite/) · [Amazon Print & Kindle](https://www.amazon.com/dp/B0HSRCN6MW)

Run the cell below (pure `NumPy`, zero GPU required) to compare **direct `float64` thin-SVD null-space editing** (`< 1e-13` collateral drift) against **naive `bfloat16` Gram SVD** (`> 4.0` Frobenius spillover into frozen memories).

In [ ]:
import numpy as np

def to_bf16_f64(x: np.ndarray) -> np.ndarray:
    u32 = np.asarray(x, dtype=np.float32).view(np.uint32)
    rb = ((u32 >> np.uint32(16)) & np.uint32(1)) + np.uint32(0x7FFF)
    return ((u32 + rb) & np.uint32(0xFFFF0000)).view(np.float32).astype(np.float64)

rng = np.random.default_rng(42)
U0, _ = np.linalg.qr(rng.standard_normal((64, 40)))
V0, _ = np.linalg.qr(rng.standard_normal((40, 40)))
K0 = (U0 * np.concatenate([np.geomspace(10.0, 0.5, 30), np.full(10, 0.03)])) @ V0.T
W0, K1, V1 = rng.standard_normal((32, 64)), rng.standard_normal((64, 4)), rng.standard_normal((32, 4))

# 1) Direct float64 thin SVD on K0 (no Gram squaring)
U, s, _ = np.linalg.svd(K0, full_matrices=False)
P_f64 = np.eye(64) - U[:, s > 1e-11 * s[0]] @ U[:, s > 1e-11 * s[0]].T
P_f64 = 0.5 * (P_f64 + P_f64.T)
K1p = P_f64 @ K1
W_f64 = W0 + ((V1 - W0 @ K1) @ np.linalg.solve(K1p @ K1p.T + 1e-6 * np.eye(64), K1p).T) @ P_f64

# 2) Naive bfloat16 Gram matrix SVD (K0 @ K0.T squares singular values)
K0_bf = to_bf16_f64(K0)
Ub, lb, _ = np.linalg.svd(to_bf16_f64(K0_bf @ K0_bf.T), full_matrices=True)
P_bf = to_bf16_f64(Ub[:, lb <= 1e-2 * lb[0]] @ Ub[:, lb <= 1e-2 * lb[0]].T)
K1b = P_bf @ K1
W_bf = W0 + ((V1 - W0 @ K1) @ np.linalg.solve(K1b @ K1b.T + 1e-6 * np.eye(64), K1b).T) @ P_bf

d64 = np.linalg.norm((W_f64 - W0) @ K0)
dbf = np.linalg.norm((W_bf - W0) @ K0)
assert d64 < 1e-12 and dbf > 1.0
print(f'Preserved-key drift ||(W_new - W0) @ K0||_F -> float64={d64:.3e} vs bf16_gram={dbf:.3e}')
